# 04 — Axial-slice maps

In [ ]:
# All paths resolve through `utils/config.py` via `result_path(family, key, fname)`.
import sys; sys.path.insert(0, "utils")
from config import *

from textwrap import fill
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from nilearn import plotting

# Four axial z-slices (mm), showing subcortical structures (matches the reference figure).
CUTS = [-26, -10, 6, 22]

brain, affine, BG = load_mask()
N_VOX = int(brain.sum())

## Reusable slice-contrast figure

In [ ]:
def _sig_T_image(family, key, pmap, thr, bonferroni=False):
    """Signed-T nifti masked by significance, or None if required files are missing.

    bonferroni=True re-derives the threshold as -log10(0.05/N_VOX) (parametric-FWE analogue); `pmap` is then the *uncorrected* parametric map.
    """
    if not result_path(family, key, F_TSTAT).exists():
        return None
    if not result_path(family, key, pmap).exists():
        return None
    T = load_map(family, key, F_TSTAT, brain=brain)
    p = load_map(family, key, pmap, brain=brain)            # -log10(p)
    use_thr = -np.log10(0.05 / N_VOX) if bonferroni else thr
    return nib.Nifti1Image(np.where(p >= use_thr, T, 0.0).astype(np.float32), affine)


def _has_sig(img):
    return img is not None and bool(np.any(img.get_fdata() != 0))


def slice_contrast_figure(family, param_pmap=F_PARAM_FDR, perm_pmap=F_PERM_FDR,
                          thr_label="p_FDR < 0.05", outname=None,
                          one_sample=False, param_bonferroni=False):
    """Axial-slice parametric and permutation figure for one analysis family.

    family           : config family key (FAM_LNM_NOCOV, FAM_VLSM, FAM_ONESAMPLE, ...)
    param_pmap       : -log10(p) map for the parametric (right, b) column
    perm_pmap        : -log10(p) map for the permutation (left, a) column
    thr_label        : colorbar/threshold label, e.g. 'p_FDR < 0.05'
    outname          : output PNG filename (saved into FIG_DIR)
    one_sample       : adjust marker/colorbar wording for a symptom-blind one-sample map
    param_bonferroni : threshold the parametric column with Bonferroni (FWE analogue)

    Returns the saved path, or None if no domain is ready (PALM unfinished).

    Only FULLY finished domains (has_results -> permutation product exists) get a row; an
    unfinished domain is omitted entirely. A finished domain with no surviving voxels in a column
    is shown as 'n.s.', so 'n.s.' always denotes a real null, never an incomplete run.
    """
    if outname is None:
        outname = f"slices_T_{family}.png"

    # Gating: which domains are fully finished?
    ready = [k for k in KEYS if has_results(family, k)]
    if not ready:
        print(f"PALM not finished for {family} — skipping {outname}")
        return None

    # Whether each column can be drawn (file present for >=1 ready domain).
    param_file = F_PARAM_UNC if param_bonferroni else param_pmap
    param_available = any(result_path(family, k, param_file).exists() for k in ready)
    perm_available = any(result_path(family, k, perm_pmap).exists() for k in ready)
    if not param_available and not perm_available:
        print(f"No significance maps ({param_file} / {perm_pmap}) for {family} yet — "
              f"skipping {outname}")
        return None
    if not param_available:
        print(f"NOTE: parametric map '{param_file}' absent for {family}; "
              f"showing permutation column only.")
    if not perm_available:
        print(f"NOTE: permutation map '{perm_pmap}' absent for {family}; "
              f"showing parametric column only.")

    # Significance-masked signed-T images per (column, ready domain). None where no voxel survives.
    perm_img, param_img = {}, {}
    for k in ready:
        perm_img[k] = (_sig_T_image(family, k, perm_pmap, THR) if perm_available else None)
        param_img[k] = (_sig_T_image(family, k, param_file, THR, bonferroni=param_bonferroni)
                        if param_available else None)

    # Shared symmetric robust T scale over all ready domains (97.5th pct of |T|).
    allT = np.abs(np.concatenate([
        load_map(family, k, F_TSTAT, brain=brain)[brain] for k in ready]))
    VMAX_T = float(np.round(np.nanpercentile(allT, 97.5)))
    if VMAX_T <= 0:
        VMAX_T = float(np.round(np.nanmax(allT))) or 1.0

    nrow = len(ready)
    fig = plt.figure(figsize=(13.5, 2.15 * nrow))
    # [row-label gap | permutation (a) | spacer | parametric (b)]
    gs = fig.add_gridspec(nrow, 4, width_ratios=[0.18, 1, 0.16, 1], wspace=0.02, hspace=0.05,
                          left=0.02, right=0.99, top=0.95, bottom=0.075)
    axes_perm = [fig.add_subplot(gs[i, 1]) for i in range(nrow)]
    axes_para = [fig.add_subplot(gs[i, 3]) for i in range(nrow)]
    fig.canvas.draw()      # finalize cell positions before drawing maps into them

    def draw(ax, img):
        if _has_sig(img):
            return plotting.plot_stat_map(
                img, bg_img=BG, display_mode="z", cut_coords=CUTS, axes=ax,
                threshold=1e-6, vmax=VMAX_T, cmap="RdBu_r", colorbar=False,
                annotate=False, black_bg=False, radiological=True)
        ax.axis("off")
        return None

    perm_disp, para_disp, perm_ns, para_ns = [], [], [], []
    for i, k in enumerate(ready):
        dp = draw(axes_perm[i], perm_img[k])           # permutation (a)
        perm_disp.append(dp)
        if dp is None:
            perm_ns.append(i)
        da = draw(axes_para[i], param_img[k])          # parametric (b)
        para_disp.append(da)
        if da is None:
            para_ns.append(i)

    # Any drawn display, to anchor headers/labels even if one column is empty.
    all_disp = [d for d in (perm_disp + para_disp) if d is not None]
    if not all_disp:
        plt.close(fig)
        print(f"No surviving voxels in any domain for {family} — nothing to plot ({outname}).")
        return None

    def slice_ycenter(disp, fallback_ax):
        try:
            ps = [cax.ax.get_position() for cax in disp.axes.values()]
            return (min(p.y0 for p in ps) + max(p.y1 for p in ps)) / 2
        except Exception:
            p = fallback_ax.get_position(); return (p.y0 + p.y1) / 2

    # Orientation markers (radiological: R on viewer's left) on first slice of the top row.
    top_disp = next((d for d in (perm_disp[0], para_disp[0]) if d is not None), None)
    if top_disp is not None:
        ax_first = list(top_disp.axes.values())[0].ax
        ax_first.text(0.03, 0.97, "R", transform=ax_first.transAxes, ha="left", va="top",
                      fontsize=9, fontweight="bold", color="black")
        ax_first.text(0.97, 0.97, "L", transform=ax_first.transAxes, ha="right", va="top",
                      fontsize=9, fontweight="bold", color="black")

    # Column headers, anchored to the brains' top across whichever columns were drawn.
    ytop = max(max(c.ax.get_position().y1 for c in d.axes.values()) for d in all_disp)
    for col, head in [(1, "a   Permutation-based"), (3, "b   Parametric")]:
        p = gs[0, col].get_position(fig)
        fig.text(p.x0, ytop + 0.008, head, fontsize=14, fontweight="bold", ha="left", va="bottom")

    # Row labels + n.s. markers, centred per row (use whichever column has a display).
    x_lab = gs[0, 1].get_position(fig).x0 - 0.008
    pperm = gs[0, 1].get_position(fig); perm_cx = (pperm.x0 + pperm.x1) / 2
    ppara = gs[0, 3].get_position(fig); para_cx = (ppara.x0 + ppara.x1) / 2
    for i in range(nrow):
        ref = perm_disp[i] or para_disp[i]
        if ref is not None:
            yc = slice_ycenter(ref, axes_perm[i])
        else:
            pa = axes_perm[i].get_position(); yc = (pa.y0 + pa.y1) / 2
        fig.text(x_lab, yc, fill(DOMAINS[ready[i]], 16), va="center", ha="right",
                 fontsize=12, fontweight="bold")
        if perm_available and i in perm_ns:
            fig.text(perm_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)
        if param_available and i in para_ns:
            fig.text(para_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)

    # Shared diverging T colorbar at the bottom.
    cax = fig.add_axes([0.40, 0.030, 0.30, 0.009])
    sm = ScalarMappable(cmap="RdBu_r", norm=Normalize(-VMAX_T, VMAX_T)); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal", extend="both",
                      ticks=[-VMAX_T, 0, VMAX_T])
    cb.set_label(rf"$T$  ({thr_label})", fontsize=10, fontweight="bold")
    cb.ax.tick_params(labelsize=9)
    for t in cb.ax.get_xticklabels():
        t.set_fontweight("bold")

    out = FIG_DIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    kind = "one-sample map" if one_sample else "group contrast"
    print(f"saved {out}  (|T| capped at {VMAX_T:.0f}; {len(ready)}/{len(KEYS)} "
          f"domains ready; {kind})")
    return out

In [ ]:
def slice_single_figure(family, pmap, kind, thr_label, outname,
                        one_sample=False, param_bonferroni=False):
    """Single-column axial-slice figure (one inference scheme) for one PALM family.

    Splits the side-by-side `slice_contrast_figure` into one standalone column, keeping the same
    aesthetics: 6 domain rows, four axial cuts CUTS=[-26,-10,6,22], radiological=True, cmap
    RdBu_r, shared symmetric robust T scale (97.5th pct of |T| over all ready domains), R/L
    markers on the first slice of the top row, n.s. label where no voxel survives, wrapped row
    labels, shared bottom diverging T colorbar.

    family           : config family key
    pmap             : -log10(p) significance map for this column
    kind             : 'perm' or 'param' (chooses the single title + Bonferroni semantics)
    thr_label        : colorbar/threshold label, e.g. r'$\\mathbf{p_{FDR}}$ < 0.05'
    outname          : output PNG filename (saved into FIG_DIR)
    one_sample       : symptom-blind one-sample wording for the saved-message
    param_bonferroni : threshold this (parametric) column with Bonferroni (FWE analogue)

    Returns the saved path, or None if no domain is ready / nothing survives.
    """
    title = "a   Permutation-based" if kind == "perm" else "a   Parametric"

    # Gating: which domains are fully finished?
    ready = [k for k in KEYS if has_results(family, k)]
    if not ready:
        print(f"PALM not finished for {family} — skipping {outname}")
        return None

    # For the Bonferroni (parametric-FWE) analogue the figure draws from the *uncorrected* map.
    map_file = F_PARAM_UNC if param_bonferroni else pmap
    available = any(result_path(family, k, map_file).exists() for k in ready)
    if not available:
        print(f"No significance map ({map_file}) for {family} yet — skipping {outname}")
        return None

    # Significance-masked signed-T images per ready domain. None where no voxel survives.
    imgs = {k: _sig_T_image(family, k, pmap, THR, bonferroni=param_bonferroni) for k in ready}

    # Shared symmetric robust T scale over all ready domains (97.5th pct of |T|).
    allT = np.abs(np.concatenate([
        load_map(family, k, F_TSTAT, brain=brain)[brain] for k in ready]))
    VMAX_T = float(np.round(np.nanpercentile(allT, 97.5)))
    if VMAX_T <= 0:
        VMAX_T = float(np.round(np.nanmax(allT))) or 1.0

    nrow = len(ready)
    fig = plt.figure(figsize=(7.0, 2.15 * nrow))
    # [row-label gap | single map column]
    gs = fig.add_gridspec(nrow, 2, width_ratios=[0.18, 1], wspace=0.02, hspace=0.05,
                          left=0.04, right=0.99, top=0.95, bottom=0.075)
    axes = [fig.add_subplot(gs[i, 1]) for i in range(nrow)]
    fig.canvas.draw()      # finalize cell positions before drawing maps into them

    def draw(ax, img):
        if _has_sig(img):
            return plotting.plot_stat_map(
                img, bg_img=BG, display_mode="z", cut_coords=CUTS, axes=ax,
                threshold=1e-6, vmax=VMAX_T, cmap="RdBu_r", colorbar=False,
                annotate=False, black_bg=False, radiological=True)
        ax.axis("off")
        return None

    disp, ns = [], []
    for i, k in enumerate(ready):
        d = draw(axes[i], imgs[k])
        disp.append(d)
        if d is None:
            ns.append(i)

    drawn = [d for d in disp if d is not None]
    if not drawn:
        plt.close(fig)
        print(f"No surviving voxels in any domain for {family} — nothing to plot ({outname}).")
        return None

    def slice_ycenter(d, fallback_ax):
        try:
            ps = [cax.ax.get_position() for cax in d.axes.values()]
            return (min(p.y0 for p in ps) + max(p.y1 for p in ps)) / 2
        except Exception:
            p = fallback_ax.get_position(); return (p.y0 + p.y1) / 2

    # Orientation markers (radiological: R on viewer's left) on first slice of the top row.
    top_disp = disp[0]
    if top_disp is not None:
        ax_first = list(top_disp.axes.values())[0].ax
        ax_first.text(0.03, 0.97, "R", transform=ax_first.transAxes, ha="left", va="top",
                      fontsize=9, fontweight="bold", color="black")
        ax_first.text(0.97, 0.97, "L", transform=ax_first.transAxes, ha="right", va="top",
                      fontsize=9, fontweight="bold", color="black")

    # Single column header, anchored to the brains' top.
    ytop = max(c.ax.get_position().y1 for d in drawn for c in d.axes.values())
    p = gs[0, 1].get_position(fig)
    fig.text(p.x0, ytop + 0.008, title, fontsize=14, fontweight="bold", ha="left", va="bottom")

    # Row labels + n.s. markers.
    x_lab = gs[0, 1].get_position(fig).x0 - 0.008
    pcol = gs[0, 1].get_position(fig); col_cx = (pcol.x0 + pcol.x1) / 2
    for i in range(nrow):
        ref = disp[i]
        if ref is not None:
            yc = slice_ycenter(ref, axes[i])
        else:
            pa = axes[i].get_position(); yc = (pa.y0 + pa.y1) / 2
        fig.text(x_lab, yc, fill(DOMAINS[ready[i]], 16), va="center", ha="right",
                 fontsize=12, fontweight="bold")
        if i in ns:
            fig.text(col_cx, yc, "n.s.", va="center", ha="center", style="italic", fontsize=12)

    # Shared diverging T colorbar at the bottom.
    cax = fig.add_axes([0.40, 0.030, 0.30, 0.009])
    sm = ScalarMappable(cmap="RdBu_r", norm=Normalize(-VMAX_T, VMAX_T)); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal", extend="both",
                      ticks=[-VMAX_T, 0, VMAX_T])
    cb.set_label(rf"$T$  ({thr_label})", fontsize=10, fontweight="bold")
    cb.ax.tick_params(labelsize=9)
    for t in cb.ax.get_xticklabels():
        t.set_fontweight("bold")

    out = FIG_DIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    cstr = "one-sample map" if one_sample else "group contrast"
    print(f"saved {out}  (|T| capped at {VMAX_T:.0f}; {len(ready)}/{len(KEYS)} "
          f"domains ready; {cstr}; {kind})")
    return out

In [ ]:
def slice_two_family_figure(left, right, outname,
                            thr=THR):
    """Two-column axial-slice figure whose columns come from DIFFERENT PALM families.

    Unlike `slice_contrast_figure` (one family, two inference schemes, ONE shared T scale), this
    draws two columns from two different families and gives each column its OWN robust symmetric T
    colour scale and its OWN bottom colorbar - needed because the one-sample T magnitudes are
    ~3-4x larger than the group-comparison T.

    left, right : (family, pmap, title) tuples. `family` is a config family key, `pmap` is the
                  -log10(p) significance map, `title` is the panel header (the a/b prefix is
                  added here).
    outname     : output PNG filename (saved into FIG_DIR).
    thr         : -log10(p) significance threshold (default THR = -log10(0.05)).

    Significance is taken from each family's own `pmap` thresholded at `thr`; the signed T comes
    from `F_TSTAT`. Aesthetics mirror the reference: 6 domain rows (DOMAINS pretty names), four
    axial cuts CUTS=[-26,-10,6,22], radiological=True with R/L markers on the top-row first slice,
    RdBu_r, wrapped row labels, per-cell n.s. where no voxel survives.

    Gated on has_results: only domains finished for BOTH families get a row; if neither column has
    any ready/available data the function prints a note and returns None.
    """
    cols = [left, right]

    # Gating: require BOTH families finished (has_results) for each plotted domain row.
    ready = [k for k in KEYS
             if has_results(left[0], k) and has_results(right[0], k)]
    if not ready:
        print(f"PALM not finished for both {left[0]} and {right[0]} - skipping {outname}")
        return None

    # Per-column availability of the requested p-map for >=1 ready domain.
    avail = [any(result_path(fam, k, pmap).exists() for k in ready)
             for fam, pmap, _ in cols]
    if not any(avail):
        print(f"No significance maps for {left[0]}/{right[0]} yet - skipping {outname}")
        return None

    # Significance-masked signed-T images per (column, ready domain). None where nothing survives.
    imgs = [{}, {}]
    for ci, (fam, pmap, _) in enumerate(cols):
        for k in ready:
            imgs[ci][k] = (_sig_T_image(fam, k, pmap, thr) if avail[ci] else None)

    # INDEPENDENT robust symmetric T scale per column (97.5th pct of |T| over ready domains).
    vmax = []
    for fam, _, _ in cols:
        allT = np.abs(np.concatenate([
            load_map(fam, k, F_TSTAT, brain=brain)[brain] for k in ready]))
        v = float(np.round(np.nanpercentile(allT, 97.5)))
        if v <= 0:
            v = float(np.round(np.nanmax(allT))) or 1.0
        vmax.append(v)

    nrow = len(ready)
    fig = plt.figure(figsize=(13.5, 2.15 * nrow))
    # [row-label gap | column a | spacer | column b]
    gs = fig.add_gridspec(nrow, 4, width_ratios=[0.18, 1, 0.16, 1], wspace=0.02, hspace=0.05,
                          left=0.02, right=0.99, top=0.95, bottom=0.10)
    axes_cols = [[fig.add_subplot(gs[i, 1]) for i in range(nrow)],
                 [fig.add_subplot(gs[i, 3]) for i in range(nrow)]]
    fig.canvas.draw()      # finalize cell positions before drawing maps into them

    def draw(ax, img, vmax_col):
        if _has_sig(img):
            return plotting.plot_stat_map(
                img, bg_img=BG, display_mode="z", cut_coords=CUTS, axes=ax,
                threshold=1e-6, vmax=vmax_col, cmap="RdBu_r", colorbar=False,
                annotate=False, black_bg=False, radiological=True)
        ax.axis("off")
        return None

    disp = [[], []]
    ns = [[], []]
    for ci in range(2):
        for i, k in enumerate(ready):
            d = draw(axes_cols[ci][i], imgs[ci][k], vmax[ci])
            disp[ci].append(d)
            if d is None:
                ns[ci].append(i)

    all_disp = [d for col in disp for d in col if d is not None]
    if not all_disp:
        plt.close(fig)
        print(f"No surviving voxels in any domain - nothing to plot ({outname}).")
        return None

    def slice_ycenter(d, fallback_ax):
        try:
            ps = [cax.ax.get_position() for cax in d.axes.values()]
            return (min(p.y0 for p in ps) + max(p.y1 for p in ps)) / 2
        except Exception:
            p = fallback_ax.get_position(); return (p.y0 + p.y1) / 2

    # Orientation markers (radiological: R on viewer's left) on first slice of the top row.
    top_disp = next((d for d in (disp[0][0], disp[1][0]) if d is not None), None)
    if top_disp is not None:
        ax_first = list(top_disp.axes.values())[0].ax
        ax_first.text(0.03, 0.97, "R", transform=ax_first.transAxes, ha="left", va="top",
                      fontsize=9, fontweight="bold", color="black")
        ax_first.text(0.97, 0.97, "L", transform=ax_first.transAxes, ha="right", va="top",
                      fontsize=9, fontweight="bold", color="black")

    # Column headers, anchored to the brains' top across whichever columns were drawn.
    ytop = max(max(c.ax.get_position().y1 for c in d.axes.values()) for d in all_disp)
    for col_gs, prefix, (fam, pmap, title) in [(1, "a", cols[0]), (3, "b", cols[1])]:
        p = gs[0, col_gs].get_position(fig)
        fig.text(p.x0, ytop + 0.008, f"{prefix}   {title}", fontsize=14, fontweight="bold",
                 ha="left", va="bottom")

    # Row labels + n.s. markers, centred per row.
    x_lab = gs[0, 1].get_position(fig).x0 - 0.008
    col_cx = [(gs[0, 1].get_position(fig).x0 + gs[0, 1].get_position(fig).x1) / 2,
              (gs[0, 3].get_position(fig).x0 + gs[0, 3].get_position(fig).x1) / 2]
    for i in range(nrow):
        ref = disp[0][i] or disp[1][i]
        if ref is not None:
            yc = slice_ycenter(ref, axes_cols[0][i])
        else:
            pa = axes_cols[0][i].get_position(); yc = (pa.y0 + pa.y1) / 2
        fig.text(x_lab, yc, fill(DOMAINS[ready[i]], 16), va="center", ha="right",
                 fontsize=12, fontweight="bold")
        for ci in range(2):
            if avail[ci] and i in ns[ci]:
                fig.text(col_cx[ci], yc, "n.s.", va="center", ha="center",
                         style="italic", fontsize=12)

    # TWO SEPARATE diverging T colorbars, one under each column, each independently labelled.
    cb_labels = [rf"$T$  ({cols[0][2]}, $p_{{FDR}}$<0.05)",
                 rf"$T$  ({cols[1][2]}, $p_{{FDR}}$<0.05)"]
    for ci, col_gs in enumerate((1, 3)):
        p = gs[0, col_gs].get_position(fig)
        cx = (p.x0 + p.x1) / 2
        cax = fig.add_axes([cx - 0.13, 0.045, 0.26, 0.009])
        sm = ScalarMappable(cmap="RdBu_r", norm=Normalize(-vmax[ci], vmax[ci])); sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal", extend="both",
                          ticks=[-vmax[ci], 0, vmax[ci]])
        cb.set_label(cb_labels[ci], fontsize=10, fontweight="bold")
        cb.ax.tick_params(labelsize=9)
        for t in cb.ax.get_xticklabels():
            t.set_fontweight("bold")

    out = FIG_DIR / outname
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"saved {out}  (left |T| cap={vmax[0]:.0f} [{left[0]}]; "
          f"right |T| cap={vmax[1]:.0f} [{right[0]}]; {len(ready)}/{len(KEYS)} domains ready)")
    return out

## FDR slice contrasts

### LNM — no covariate (`lnm_nocov`)

LNM group comparison (impaired vs unimpaired), only with cohort fixed effects and within-cohort exchangeability blocks. Blue =
impaired < unimpaired functional connectivity to the lesion network; red = impaired > unimpaired.

In [ ]:
slice_contrast_figure(CORE_FAMILY, param_pmap=F_PARAM_FDR, perm_pmap=F_PERM_FDR,
                      thr_label=r"$\mathbf{p_{FDR}}$ < 0.05",
                      outname=f"slices_T_{CORE_FAMILY}.png")

#### VLSM (`vlsm_volcov`)

Voxel-based lesion–symptom mapping analogue of the same impaired-vs-unimpaired contrast
(volume-adjusted), run on the lesion masks rather than the connectivity maps.

In [ ]:
slice_contrast_figure(FAM_VLSM, param_pmap=F_PARAM_FDR, perm_pmap=F_PERM_FDR,
                      thr_label=r"$\mathbf{p_{FDR}}$ < 0.05",
                      outname=f"slices_T_{FAM_VLSM}.png")

#### One-sample LNM (`onesample`) — symptom-blind benchmark

One-sample map of the impaired cases' lesion-network connectivity, not a group
contrast.

In [ ]:
slice_single_figure(FAM_ONESAMPLE, F_PARAM_FDR, kind="param",
                      thr_label=r"$\mathbf{p_{FDR}}$ < 0.05",
                      outname=f"slices_T_{FAM_ONESAMPLE}.png", one_sample=True)

## Individual (split) FDR slice figures — permutation-only and parametric-only

In [ ]:
PARAM_SINGLE_FAMILIES = [FAM_LNM_NOCOV, FAM_VLSM]
for fam in GROUP_FAMILIES:
    slice_single_figure(fam, F_PERM_FDR, kind='perm',
                        thr_label=r'$\mathbf{p_{FDR}}$ < 0.05',
                        outname=f'slices_T_{fam}_perm.png')
    if fam in PARAM_SINGLE_FAMILIES:
        slice_single_figure(fam, F_PARAM_FDR, kind='param',
                            thr_label=r'$\mathbf{p_{FDR}}$ < 0.05',
                            outname=f'slices_T_{fam}_param.png')

## FWE variant

### LNM — no volume covariate (`lnm_nocov`, FWE)

In [ ]:
slice_contrast_figure(CORE_FAMILY, param_pmap=F_PARAM_FWE, perm_pmap=F_PERM_FWE,
                      thr_label=r"$\mathbf{p_{FWE}}$ < 0.05", param_bonferroni=True,
                      outname=f"slices_T_{CORE_FAMILY}_FWE.png")